# 06 — 驗證 MNIST 線上推論（單元 B-7）

在 **Workbench 內 Run cell**，呼叫已部署的 KServe V2 API。

| 項目 | 值 |
|------|-----|
| 部署名（Service） | 預設 `mnist-classifier-elyra` |
| API model 名 | **可能與部署名不同**（ONNX/OVMS 常見）；本 Notebook 會自動查 `/v2/models` |
| 前置 | Dashboard 上該部署 Status = **Ready** |

> `Connection refused` → port 問題（自動探測）  
> `Model ... not found` (404) → **API 模型名**不對（自動列出後再試）


## 步驟 1：確認部署（Dashboard）

1. **Models** → **`mnist-classifier-elyra`** → **Ready**
2. （可選）複製 Inference endpoint 到下方 `URL_OVERRIDE`


## 步驟 2：設定目標


In [ ]:
import json
import os
import socket
import urllib.error
import urllib.request
from pathlib import Path

NAMESPACE = os.environ.get("NB_NAMESPACE") or os.environ.get("NAMESPACE") or "rhoai-quickstart"
# 這是 InferenceService / Service 名稱（DNS 用）
DEPLOYMENT = os.environ.get("MODEL", "mnist-classifier-elyra")

# 若已知 V2 API 模型名可填；留空則自動發現
API_MODEL_OVERRIDE = os.environ.get("API_MODEL", "").strip()

URL_OVERRIDE = os.environ.get("URL", "").strip()

HOST = f"{DEPLOYMENT}-predictor.{NAMESPACE}.svc.cluster.local"
CANDIDATE_PORTS = [80, 8080, 8888, 8008]

token_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
TOKEN = os.environ.get("TOKEN") or (token_path.read_text().strip() if token_path.exists() else "")

print(f"NAMESPACE={NAMESPACE}")
print(f"DEPLOYMENT={DEPLOYMENT}")
print(f"HOST={HOST}")
print(f"TOKEN length={len(TOKEN)}")
assert TOKEN, "找不到 ServiceAccount token；請在 Workbench 內執行本 Notebook"


## 步驟 3：探測可連線的 base URL


In [ ]:
def tcp_open(host: str, port: int, timeout: float = 2.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def normalize_base(url: str) -> str:
    u = url.rstrip("/")
    if "/v2/" in u:
        u = u.split("/v2/")[0]
    return u.rstrip("/")


def http_json(url: str, method: str = "GET", data=None):
    body = None if data is None else json.dumps(data).encode()
    headers = {"Authorization": f"Bearer {TOKEN}"}
    if body is not None:
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=60) as resp:
        raw = resp.read()
        return resp.status, (json.loads(raw) if raw else None)


bases = []
if URL_OVERRIDE:
    bases.append(normalize_base(URL_OVERRIDE))
else:
    for port in CANDIDATE_PORTS:
        if tcp_open(HOST, port):
            bases.append(f"http://{HOST}" if port == 80 else f"http://{HOST}:{port}")
            print(f"TCP OK: {HOST}:{port}")
        else:
            print(f"TCP refused/closed: {HOST}:{port}")

if not bases:
    raise RuntimeError(
        "沒有任何 port 可連線。請確認部署 Ready，且 DEPLOYMENT 名稱正確。"
    )

# 選第一個能回應 /v2 或 /v2/models 的 base
BASE = None
for b in bases:
    for path in ("/v2/models", "/v2/health/ready", "/v2"):
        try:
            status, _ = http_json(f"{b}{path}")
            print(f"HTTP {status} {b}{path}")
            BASE = b
            break
        except Exception as e:
            print(f"fail {b}{path}: {e}")
    if BASE:
        break

if not BASE:
    BASE = bases[0]
    print(f"Fallback BASE={BASE}")
else:
    print(f"Using BASE={BASE}")


## 步驟 4：查出 V2 API 模型名稱並推論


In [ ]:
def list_models(base: str) -> list[str]:
    names = []
    try:
        _, payload = http_json(f"{base}/v2/models")
        print("/v2/models =>", payload)
        if isinstance(payload, dict):
            # OVMS / KServe 可能回 {"models":[...]} 或 {"data":[...]}
            for key in ("models", "data"):
                if key in payload and isinstance(payload[key], list):
                    for item in payload[key]:
                        if isinstance(item, str):
                            names.append(item)
                        elif isinstance(item, dict) and "name" in item:
                            names.append(item["name"])
            if "name" in payload:
                names.append(payload["name"])
        elif isinstance(payload, list):
            for item in payload:
                if isinstance(item, str):
                    names.append(item)
                elif isinstance(item, dict) and "name" in item:
                    names.append(item["name"])
    except Exception as e:
        print(f"list /v2/models failed: {e}")
    # 去重保序
    seen = set()
    out = []
    for n in names:
        if n not in seen:
            seen.add(n)
            out.append(n)
    return out


candidates = []
if API_MODEL_OVERRIDE:
    candidates.append(API_MODEL_OVERRIDE)
candidates.extend(list_models(BASE))
# 常見後備名稱（OVMS / Dashboard ONNX）
for n in (DEPLOYMENT, "model", "mnist-classifier", "mnist", "mnist-onnx"):
    if n not in candidates:
        candidates.append(n)

print("Model name candidates:", candidates)

payload = {
    "inputs": [
        {
            "name": "input",
            "shape": [1, 1, 28, 28],
            "datatype": "FP32",
            "data": [0.0] * 784,
        }
    ]
}

last_err = None
result = None
used = None
api_model = None

for name in candidates:
    paths = [
        f"{BASE}/v2/models/{name}/infer",
        f"{BASE}/v2/models/{name}/versions/1/infer",
    ]
    done = False
    for infer_url in paths:
        try:
            status, result = http_json(infer_url, method="POST", data=payload)
            used = infer_url
            api_model = name
            done = True
            break
        except urllib.error.HTTPError as e:
            body = e.read().decode(errors="replace")
            last_err = f"{infer_url} -> HTTP {e.code}: {body}"
            print(last_err)
        except Exception as e:
            last_err = f"{infer_url} -> {e}"
            print(last_err)
    if done:
        break

if result is None:
    raise RuntimeError(
        "推論失敗。請把上面 /v2/models 的輸出貼給講師。\n"
        f"最後錯誤: {last_err}"
    )

scores = result["outputs"][0]["data"]
pred = max(range(len(scores)), key=lambda i: scores[i])
print(f"API model name: {api_model}")
print(f"URL: {used}")
print(f"Predicted digit: {pred}")
print(f"scores[:5]={scores[:5]}")
print("B-7 online inference OK.")


## 完成檢查 / 排查

- [ ] 印出 `Predicted digit: N` 與 `B-7 online inference OK.`

| 症狀 | 處理 |
|------|------|
| Connection refused | port；本 Notebook 會探測 |
| `Model ... is not found` | API 模型名 ≠ 部署名；看步驟 4 的 `/v2/models`，或設 `API_MODEL` |
| 401/403 | 在 Workbench 內用 SA token 執行 |

已知可用時可固定：

```python
API_MODEL_OVERRIDE = "實際名稱"  # 從 /v2/models 看到的 name
```
